# PatchCore AI 학습 노트: hc- sr04 모듈 back



### colab에 zip 파일(모듈 back 사진) 업로드

In [1]:
from google.colab import files
uploaded = files.upload()   # hc_sr04_back.zip 선택

Saving hc_sr04_back.zip to hc_sr04_back.zip


In [2]:
!unzip -q hc_sr04_back.zip -d /content/pcb_dataset
!ls /content/pcb_dataset/hc_sr04_back/normal | wc -l
!ls /content/pcb_dataset/hc_sr04_back/abnormal | wc -l

400
63


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [5]:
!pip install -q "anomalib[full,cu126]"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.7/829.7 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.7 MB/s eta 0:00:00
   ━━━

### 1. Folder datamodule

In [6]:
from anomalib.data import Folder

datamodule_back = Folder(
    name="hc_sr04_back",
    root="/content/pcb_dataset/hc_sr04_back",
    normal_dir="normal",
    abnormal_dir="abnormal",
    normal_split_ratio=0.2,
    train_batch_size=8,
    eval_batch_size=8,
    num_workers=2,
)
datamodule_back.setup()

### 2. 모델 + 엔진 (resnet18 + fp16, 검증된 조합 그대로)

In [7]:
from anomalib.models import Patchcore
from anomalib.engine import Engine

model_01_back = Patchcore(
    backbone="resnet18",
    layers=["layer2", "layer3"],
    coreset_sampling_ratio=0.15,
)

engine_01_back = Engine(
    max_epochs=1,
    accelerator="gpu",
    enable_progress_bar=False,
    default_root_dir="/content/results/hc_sr04_back_model_01",
)

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

### 3. 학습

In [8]:
engine_01_back.fit(model=model_01_back, datamodule=datamodule_back)

INFO:lightning_fabric.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:lightning_fabric.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:lightning_fabric.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │  2.8 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.131                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/loops/fit_loop.py:538: Found 69 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.
Selecting Coreset Indices.: 100%|██████████| 49151/49151 [04:15<00:00, 192.02it/s]
INFO:lightning_fabric.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


### 4. 평가

In [9]:
engine_01_back.test(model=model_01_back, datamodule=datamodule_back)

INFO:lightning_fabric.utilities.rank_zero:The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric AUROC was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric F1Score was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            1.0            │
│       image_F1Score       │    0.9841269850730896     │
└───────────────────────────┴───────────────────────────┘

[{'image_AUROC': 1.0, 'image_F1Score': 0.9841269850730896}]

### 5. score 분포 확인(이상치 체크)

In [10]:
import numpy as np
from anomalib.data import PredictDataset

normal_dir = "/content/pcb_dataset/hc_sr04_back/normal"
predict_dataset = PredictDataset(path=normal_dir, image_size=(256, 256))
predictions = engine_01_back.predict(model=model_01_back, dataset=predict_dataset)

scores = [float(p.pred_score[0]) for p in predictions]
print("정상 score 평균:", np.mean(scores))
print("정상 score 최댓값:", np.max(scores))
print("정상 score 분포:", np.percentile(scores, [50, 90, 95, 99]))

INFO:lightning_fabric.utilities.rank_zero:The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


정상 score 평균: 0.0006931885331869126
정상 score 최댓값: 0.07484057545661926
정상 score 분포: [0.         0.         0.         0.03167508]


In [11]:
abnormal_dir = "/content/pcb_dataset/hc_sr04_back/abnormal"
abnormal_dataset = PredictDataset(path=abnormal_dir, image_size=(256, 256))
abnormal_predictions = engine_01_back.predict(model=model_01_back, dataset=abnormal_dataset)
abnormal_scores = [float(p.pred_score[0]) for p in abnormal_predictions]
print("불량 score 목록:", sorted(abnormal_scores))

INFO:lightning_fabric.utilities.rank_zero:The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


불량 score 목록: [0.4853368103504181, 0.5000002384185791, 0.5004624128341675, 0.509434163570404, 0.5128275156021118, 0.5239087343215942, 0.5418972373008728, 0.5494787693023682, 0.5519531965255737, 0.5577805638313293, 0.5601842403411865, 0.5618045330047607, 0.5641288161277771, 0.5683344602584839, 0.5693281888961792, 0.5742872953414917, 0.5756455659866333, 0.5774104595184326, 0.5907518863677979, 0.5983816385269165, 0.6028683185577393, 0.6140851974487305, 0.6294270753860474, 0.6339075565338135, 0.638129711151123, 0.6414989233016968, 0.6427905559539795, 0.6557744145393372, 0.6875725984573364, 0.6945040225982666, 0.6948994994163513, 0.7025352716445923, 0.727523684501648, 0.7290118336677551, 0.7310318350791931, 0.7349709272384644, 0.7411285042762756, 0.7482205033302307, 0.7601514458656311, 0.7605550289154053, 0.7616906762123108, 0.769174337387085, 0.7760599851608276, 0.7789694666862488, 0.7945265769958496, 0.7957114577293396, 0.7963855862617493, 0.7993700504302979, 0.8043571710586548, 0.80882692

### Export


In [14]:
from anomalib.deploy import ExportType
import shutil
from pathlib import Path

engine_01_back.export(
    model=model_01_back,
    export_type=ExportType.ONNX,
    export_root="/content/export/hc_sr04_back_model_01",
)

DRIVE_BACKUP = "/content/drive/MyDrive/AI_MachineLearning/boardguard"
export_dest = Path(DRIVE_BACKUP) / "export_hc_sr04_back_model_01"
if export_dest.exists():
    shutil.rmtree(export_dest)
shutil.copytree("/content/export/hc_sr04_back_model_01", export_dest)

PosixPath('/content/drive/MyDrive/AI_MachineLearning/boardguard/export_hc_sr04_back_model_01')

In [15]:
import os
path = "/content/drive/MyDrive/AI_MachineLearning/boardguard/export_hc_sr04_back_model_01"
print(os.path.exists(path))
print(os.listdir(path))

True
['weights']


In [18]:
from google.colab import files
files.download("/content/export/hc_sr04_back_model_01/weights/onnx/model.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>